# HW0 — Introduction & Cost of Living Analysis
**Sahil Bains**

---
## Part 1 — About Me

### Why are you pursuing the MSBA program?

I'm pursuing the MSBA program because I want a stronger foundation in analytics and machine learning. I also want to get better at turning analysis into something useful for decision-making. The program feels like a good way to connect what I learned before with the kind of product analytics work I want to do next.

### What did you do before entering the program?

Before starting the program, I graduated from the University of Washington, where I studied Data Science and GIS. After that, I worked full-time for a bit and saved money for grad school. That experience helped confirm that I want to keep working in data-focused roles.

### What type of role, industry, or company are you interested in after graduation?

After graduation, I'm mainly interested in data scientist roles in product analytics. I like work that uses customer, marketing, and product usage data to understand churn and adoption. Right now I'm most interested in tech companies and startups in the Bay Area, Los Angeles, or Bellevue.

### What topics or methods are you most interested in learning in this course?

In this course, I want to get more comfortable with machine learning methods that are actually used in practice. I'm especially interested in supervised learning, model evaluation, and interpretation. I also want more hands-on practice so I can feel better prepared for technical interviews.

---
## Part 2 — Cost of Living & Salary Analysis

In this section, I estimate my monthly living costs in San Francisco, Los Angeles, and Bellevue. Then I compare those costs with a target salary of $140,000 to see which city would feel most affordable.

The numbers are rough estimates based on recent rent, cost-of-living, and salary data. I kept the assumptions together below so I can change them and rerun everything easily.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

### 2.1 — Key Inputs

I put the main assumptions here so it is easy to change them and rerun the rest of the notebook.

In [ ]:
# Main assumptions
desired_salary = 140_000
base_tax_rate = 0.22
savings_buffer = 0.10

# Approximate state tax rates
state_tax = {
    "San Francisco": 0.093,
    "Los Angeles":   0.093,
    "Bellevue":      0.000,
}

# Estimated monthly costs for one person in each city
costs = {
    "San Francisco": {
        "rent":               3100,
        "groceries":           450,
        "dining_out":          400,
        "transportation":      180,
        "utilities":           150,
        "health_insurance":    200,
        "entertainment_misc":  350,
    },
    "Los Angeles": {
        "rent":               2300,
        "groceries":           400,
        "dining_out":          350,
        "transportation":      350,
        "utilities":           140,
        "health_insurance":    200,
        "entertainment_misc":  300,
    },
    "Bellevue": {
        "rent":               2100,
        "groceries":           380,
        "dining_out":          300,
        "transportation":      280,
        "utilities":           130,
        "health_insurance":    200,
        "entertainment_misc":  250,
    },
}

locations = list(costs.keys())
categories = list(next(iter(costs.values())).keys())

print("Cities:", locations)
print("Cost categories:", categories)

### 2.2 — Build the Cost-of-Living DataFrame

In [ ]:
df_costs = pd.DataFrame(costs).T
df_costs.index.name = "City"

df_costs["total_monthly"] = df_costs[categories].sum(axis=1)
df_costs["total_annual"]  = df_costs["total_monthly"] * 12

print(df_costs[["rent", "total_monthly", "total_annual"]].to_string())

### 2.3 — Salary Calculations

For each city, I wanted to answer two questions:
1. What base salary would I need to cover expenses and still save 10% of take-home pay?
2. If I make $140k, how much money is left over each month after expenses?

In [ ]:
results = []

for city in locations:
    monthly_expenses = df_costs.loc[city, "total_monthly"]
    annual_expenses = monthly_expenses * 12
    effective_tax_rate = base_tax_rate + state_tax[city]
    after_tax_share = 1 - effective_tax_rate

    required_takehome_annual = annual_expenses / (1 - savings_buffer)
    required_takehome_mo = required_takehome_annual / 12
    min_gross_salary = required_takehome_annual / after_tax_share

    desired_takehome_mo = desired_salary * after_tax_share / 12
    monthly_surplus = desired_takehome_mo - monthly_expenses

    results.append({
        "city": city,
        "monthly_expenses": monthly_expenses,
        "annual_expenses": annual_expenses,
        "effective_tax_rate": round(effective_tax_rate, 3),
        "required_takehome_mo": round(required_takehome_mo, 2),
        "min_gross_salary": round(min_gross_salary, -2),
        "desired_salary": desired_salary,
        "desired_takehome_mo": round(desired_takehome_mo, 2),
        "monthly_surplus": round(monthly_surplus, 2),
    })

df_results = pd.DataFrame(results).set_index("city")
df_results.index.name = "City"

display_cols = [
    "monthly_expenses",
    "required_takehome_mo",
    "effective_tax_rate",
    "min_gross_salary",
    "desired_takehome_mo",
    "monthly_surplus",
]
print(df_results[display_cols].to_string())

### 2.4 — Visualization

The bars show the monthly cost breakdown in each city. The green circles show take-home pay at $140k, and the orange diamonds show the monthly take-home needed to cover expenses plus the savings buffer.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
          "#8172B2", "#937860", "#DA8BC3"]
x = np.arange(len(locations))
bottom = np.zeros(len(locations))

for i, category in enumerate(categories):
    values = np.array([costs[city][category] for city in locations])
    ax.bar(
        x,
        values,
        bottom=bottom,
        color=colors[i],
        label=category.replace("_", " ").title(),
        width=0.55,
    )
    bottom += values

desired_takehome = df_results.loc[locations, "desired_takehome_mo"].to_numpy()
required_takehome = df_results.loc[locations, "required_takehome_mo"].to_numpy()

ax.scatter(x, desired_takehome, color="green", s=90, zorder=3,
           label="Take-home at $140k")
ax.scatter(x, required_takehome, color="darkorange", marker="D", s=70,
           zorder=3, label="Take-home needed")

for i, total in enumerate(bottom):
    ax.text(x[i], total + 90, f"${total:,.0f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(locations, fontsize=12)
ax.set_ylabel("Monthly ($)", fontsize=11)
ax.set_title("Monthly Cost of Living vs. Take-Home Pay by City",
             fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.set_ylim(0, max(bottom.max(), desired_takehome.max(), required_takehome.max()) * 1.18)

plt.tight_layout()
plt.savefig("cost_of_living.png", dpi=150)
plt.show()

### 2.5 — Summary & Interpretation

In [ ]:
summary_table = df_results[[
    "monthly_expenses",
    "required_takehome_mo",
    "desired_takehome_mo",
    "monthly_surplus",
    "min_gross_salary",
]].rename(columns={
    "monthly_expenses": "Monthly expenses",
    "required_takehome_mo": "Take-home needed / month",
    "desired_takehome_mo": "Take-home at $140k / month",
    "monthly_surplus": "Left over each month",
    "min_gross_salary": "Minimum gross salary",
})

print(summary_table.to_string())
print()
print(f"Target salary used: ${desired_salary:,}")
print(f"Savings buffer used: {savings_buffer:.0%}")
print("Washington state tax: 0% | California state tax used: 9.3%")

**Quick notes:**

- In this setup, $140k works in all three cities, but the amount left over each month is very different.
- Bellevue looks best financially because rent is lower and there is no state income tax.
- Los Angeles is in the middle.
- San Francisco looks the hardest to afford on base salary alone because rent is the highest.
- If I change the values in Section 2.1 and rerun the notebook, the table and plot update automatically.

---
## Part 3 — Reflection

### Did your notebook run successfully from start to finish?

Yes. I restarted the kernel and ran the notebook from top to bottom without errors. The tables and chart all showed up correctly.

### How long did this assignment take you?

Around 2 hours. Most of the time went into choosing reasonable cost and salary estimates for each city.

### One technical skill you feel confident about, and one you want to strengthen

**Confident in:** Using `pandas` to organize data and build summary tables.

**Want to strengthen:** Model evaluation and interpretability. I want to get better at understanding why a model makes certain predictions and how to explain that clearly.